# Introduction
  
This project aims to sort through different lead datasets. The file layout is as follows:  

- 82M Part-2  
- 500 Million B2C Leads Database  
- Apollo  
- Crypto Emails  
- Ecom Owners 100k  
- Linkedin L-Series  
- Python CSV Processing Folder  
- Real Estate Agents  
- States Divided  
- Zoom Info 70 Million

The main goal is to produce a master sql database to query, including which file/dataset each lead came from.

I will first ensure each file is turned into a csv file for faster loading.

End Program should only take 1 parameter: Search quiery and output master csv file.

End file information:  
list the information of which file directory it came from
Include key contact features, such as names, phone number/company number, email address, linkedin.

# Master SQL

Todo:
- Convert all Excel files to CSV format  
- Search for the term "plumb" in all datasets  
- Analyze column differences across datasets  
- Rename columns with inconsistent naming  
- Combine filtered results into a master CSV file  
- [] Document processing times for each dataset


# CSV Functions:

## Excel to CSV:

In [ ]:
from xlsx2csv import Xlsx2csv
import time, os, glob

def xcl_to_csv(folder_file_path, index_start=1):
    """
    Convert Excel (.xlsx) files in a specified folder to CSV format.

    This function processes all Excel files in the given folder, converting them to CSV format.
    It includes error handling to skip corrupt or problematic files and allows resuming from a specific file index.

    Args:
    folder_file_path (str): Path to the folder containing Excel files.
    index_start (int, optional): The file index to start processing from. Defaults to 1.

    Returns:
    None

    Note:
    - The function uses the Xlsx2csv library for conversion.
    - Converted CSV files are saved in the same folder as the original Excel files.
    - Progress and timing information is printed for each file processed.
    - Files that cause errors during conversion are skipped, and the error is logged.
    """

    # Get all Excel files in the specified folder
    files = glob.glob(folder_file_path + "/*.xlsx")
    print(f"Files in folder: {len(files)}\n")
    
    for index, file in enumerate(files):
        # Skip files before the specified start index
        if index < index_start - 1:
            continue

        print(f'Current file {index + 1}: {file}')
        percent = (index + 1) / len(files) * 100
        
        # Extract file name without extension
        file_name = os.path.basename(file).rstrip('.xlsx')
        
        start_time = time.time()
        
        try:
            # Convert Excel file to CSV
            Xlsx2csv(file).convert(f'{folder_file_path}/{file_name}.csv')
            
            end_time = time.time()
            print(f"File processed in: {end_time - start_time:.2f} seconds")
        except Exception as e:
            print(f"Error processing file {file}: {str(e)}")
            print("Skipping to next file...\n")
            continue  # Skip to the next file
        
        print(f'{percent:.2f}% complete\n')

# Usage example:
# xcl_to_csv("/path/to/excel/files")
# To start from a specific file (e.g., the 10th file):
# xcl_to_csv("/path/to/excel/files", index_start=10)

## Pandas General File Search:

In [ ]:
"""
CSV File Search and Filter Utility

This script provides functions to search for a specific term across multiple CSV files,
filter the matching rows, and save the results to new CSV files.

Functions:
    file_encoding(file_path): Detect the encoding of a given file.
    search_files(folder_file_path, search_term, file_index_start=1, encoding="utf-8", subfolder_search=False, output_folder=None):
        Search for a term in CSV files and save filtered results.

Dependencies:
    - pandas
    - charset_normalizer
    - glob
    - os
    - time

Note: This is described as the "ORIGINAL SCRIPT" in the comments.
"""

import glob, os, time
import pandas as pd
from charset_normalizer import detect

def file_encoding(file_path):
    """
    Detect the encoding of a given file.

    Args:
        file_path (str): Path to the file.

    Returns:
        str: Detected encoding of the file.

    This function reads the file in chunks and uses charset_normalizer to detect the encoding.
    It stops after reading 10 MB of data or the end of the file, whichever comes first.
    """
    CHUNK_SIZE = 1024 * 1024  # Read in 1 MB chunks
    raw_data = b""  # Empty bytes object to accumulate chunks
    with open(file_path, "rb") as f:
        while chunk := f.read(CHUNK_SIZE):  # Read in chunks
            raw_data += chunk
            if len(raw_data) > 10 * CHUNK_SIZE:  # Break after reading 10 MB
                break  # Stop after processing enough data to determine encoding
    result = detect(raw_data)
    print(result)
    print(result["encoding"])
    return result["encoding"]

def search_files(folder_file_path, search_term, file_index_start=1, encoding="utf-8", subfolder_search=False, output_folder=None):
    """
    Search for a term in CSV files, filter matching rows, and save results.

    Args:
        folder_file_path (str): Path to the folder containing CSV files.
        search_term (str): Term to search for in the CSV files.
        file_index_start (int, optional): Index of the file to start processing from. Defaults to 1.
        encoding (str, optional): Encoding of the CSV files. Defaults to "utf-8".
        subfolder_search (bool, optional): Whether to search in subfolders. Defaults to False.
        output_folder (str, optional): Path to save results. If None, creates a folder in the input directory.

    This function processes CSV files in the specified folder (and subfolders if enabled),
    searches for the given term in each file, filters rows containing the term,
    and saves the filtered data to new CSV files in the output folder.

    The function uses pandas for CSV processing and provides progress updates and timing information.

    Note:
        - The search is case-insensitive.
        - The function skips bad lines in CSV files.
        - Results are saved with filenames prefixed by the search term.
    """
    if subfolder_search:
        files = glob.glob(folder_file_path + "/**/*.csv", recursive=True)
    else:
        files = glob.glob(folder_file_path + "/*.csv")  # Grab all CSV files in folder
        
    print(f"Files in folder: {len(files)}")
    print(f"Search term: {search_term}")
    output_folder = output_folder or f"{folder_file_path}/{search_term}_results"
    initial_time = time.time()
    
    # Check if the folder exists before creating it
    if not os.path.exists(output_folder):
        print(f"Creating output folder: {output_folder}\n")
        os.mkdir(output_folder)
    else:
        print(f"Output folder already exists: {output_folder}\n")
    
    for index, file in enumerate(files):
        if index < file_index_start - 1:
            continue
        print(f'Current file {index + 1}: {file}')
        percent = (index + 1) / len(files) * 100
        start_time = time.time()
        
        # Load CSV file with pandas
        df = pd.read_csv(file, on_bad_lines='skip', encoding=encoding)
        
        # Filter rows based on whether the search term exists in any cell (non-case sensitive)
        string_columns = df.select_dtypes(include=["object", "string"]).columns
        mask = df[string_columns].apply(lambda col: col.str.contains(search_term, case=False, na=False))
        filtered_df = df[mask.any(axis=1)]
        
        # Save the filtered DataFrame to a new CSV file
        filtered_df.to_csv(f"{output_folder}/{search_term}_{os.path.basename(file)}", index=False)
        
        end_time = time.time()
        print(f"File processed in: {end_time - start_time:.2f} seconds")
        print(f'{percent:.2f}% complete\n')
    
    total_time = (time.time() - initial_time)
    print(f"\nAll files processed in {total_time:.2f} seconds.\n")

# Usage example:
# search_files("/path/to/csv/files", "search_term")

## Apollo Custom Formulas:

In [21]:
import pandas as pd
import csv
import glob
import os

def load_chunks(file_path):
    chunks = pd.read_csv(file_path, chunksize=100000, on_bad_lines='skip', encoding='utf-8',engine='python', quoting=csv.QUOTE_NONE, sep = '\t') # TAB SEPERATOR
    return chunks

def search_chunks(chunks, search_term, output_folder, file):
    filtered_chunks = []  # Store filtered chunks
    for chunk in chunks:
        # Does not search columns with mixed data types, which are throwing errors
        # print(chunk.dtypes)
        string_columns = chunk.select_dtypes(include=["object", "string"]).columns 
        mask = chunk[string_columns].apply(lambda col: col.str.contains(search_term, case=False, na=False))
        filtered_chunk = chunk[mask.any(axis=1)]
        if not filtered_chunk.empty:
            filtered_chunks.append(filtered_chunk)
    
    # Combine all filtered chunks into one DataFrame if there are any filtered chunks
    if filtered_chunks:
        print(len(filtered_chunks))
        filtered_df = pd.concat(filtered_chunks, ignore_index=True)
        if not filtered_df.empty:
            output_file = f"{output_folder}/{search_term}_{os.path.basename(file)}.csv"
            filtered_df.to_csv(output_file, index=False)
            print(f"Filtered data saved to {output_file}")
        else:
            print("No matching data found in this file.")

## 82M Part-2  
- All files types are excel files.

### Excel to CSV

In [ ]:
path = "Data/82M Part-2"
files = glob.glob(path + "/*.xlsx")

print(len(files))
# 30, all files loaded

In [ ]:
from xlsx2csv import Xlsx2csv
import time

def aggregate_data(files, starting_index):
    for index, file in enumerate(files):

        # add starting point for it to run by
        if index < starting_index - 1:
            continue

        print(f'Current file {index + 1}: {file}')
        percent = (index + 1) / len(files) * 100
        print(f'{percent:.2f}% complete')

        file_name = f'list_xl {index + 1}'
        start_time = time.time()

        Xlsx2csv(file).convert(f'Data/82M Part-2/{file_name}.csv') # Output path
        end_time = time.time()
        print(f"pandas (chunked) time: {end_time - start_time} seconds")

aggregate_data(files,29)

### Logs

Current file 5: Data/82M Part-2\List # 05_1,032,908 Contacts New Project 82 Million Part-2.xlsx  
16.67% complete  
pandas (chunked) time: 69.94514632225037 seconds  

Current file 6: Data/82M Part-2\List # 06_1,034,732 Contacts New Project 82 Million Part-2.xlsx  
20.00% complete  
pandas (chunked) time: 70.76683497428894 seconds  

Current file 7: Data/82M Part-2\List # 07_1,037,389 Contacts New Project 82 Million Part-2.xlsx  
23.33% complete  
pandas (chunked) time: 74.39700222015381 seconds  

Current file 8: Data/82M Part-2\List # 08_1,032,116 Contacts New Project 82 Million Part-2.xlsx  
26.67% complete  
pandas (chunked) time: 74.39258241653442 seconds  

Current file 9: Data/82M Part-2\List # 09_1,035,431 Contacts New Project 82 Million Part-2.xlsx  
30.00% complete  
pandas (chunked) time: 69.61842250823975 seconds  

Current file 10: Data/82M Part-2\List # 10_1,035,364 Contacts New Project 82 Million Part-2.xlsx  
33.33% complete  
pandas (chunked) time: 70.59091281890869 seconds  

Current file 11: Data/82M Part-2\List # 11_1,042,138 Contacts New Project 82 Million Part-2.xlsx  
36.67% complete  
pandas (chunked) time: 70.59927225112915 seconds  

Current file 12: Data/82M Part-2\List # 12_1,037,256 Contacts New Project 82 Million Part-2.xlsx  
40.00% complete  
pandas (chunked) time: 70.96576690673828 seconds  

Current file 13: Data/82M Part-2\List # 13_1,039,816 Contacts New Project 82 Million Part-2.xlsx  
43.33% complete  
pandas (chunked) time: 70.6404812335968 seconds  

Current file 14: Data/82M Part-2\List # 14_1,038,206 Contacts New Project 82 Million Part-2.xlsx  
46.67% complete  
pandas (chunked) time: 70.96624755859375 seconds  

Current file 15: Data/82M Part-2\List # 15_1,035,405 Contacts New Project 82 Million Part-2.xlsx  
50.00% complete  
pandas (chunked) time: 69.68088459968567 seconds  

Current file 16: Data/82M Part-2\List # 16_1,034,217 Contacts New Project 82 Million Part-2.xlsx  
53.33% complete  
pandas (chunked) time: 70.73270750045776 seconds  

Current file 17: Data/82M Part-2\List # 17_1,035,819 Contacts New Project 82 Million Part-2.xlsx  
56.67% complete  
pandas (chunked) time: 69.78206658363342 seconds  

Current file 18: Data/82M Part-2\List # 18_1,037,312 Contacts New Project 82 Million Part-2.xlsx  
60.00% complete  
pandas (chunked) time: 69.60776662826538 seconds  

Current file 19: Data/82M Part-2\List # 19_1,034,806 Contacts New Project 82 Million Part-2.xlsx  
63.33% complete  
pandas (chunked) time: 69.14107990264893 seconds  

Current file 20: Data/82M Part-2\List # 20_1,033,150 Contacts New Project 82 Million Part-2.xlsx  
66.67% complete  
pandas (chunked) time: 69.88297748565674 seconds  

Current file 21: Data/82M Part-2\List # 21_1,035,777 Contacts New Project 82 Million Part-2.xlsx  
70.00% complete  
pandas (chunked) time: 69.59670186042786 seconds  

Current file 22: Data/82M Part-2\List # 22_1,036,334 Contacts New Project 82 Million Part-2.xlsx  
73.33% complete  
pandas (chunked) time: 69.73396253585815 seconds  

Current file 23: Data/82M Part-2\List # 23_1,036,165 Contacts New Project 82 Million Part-2.xlsx  
76.67% complete  
pandas (chunked) time: 70.29383778572083 seconds  

Current file 24: Data/82M Part-2\List # 24_1,046,530 Contacts New Project 82 Million Part-2.xlsx  
80.00% complete  
pandas (chunked) time: 70.16770935058594 seconds  

Current file 25: Data/82M Part-2\List # 25_1,038,473 Contacts New Project 82 Million Part-2.xlsx  
83.33% complete  
pandas (chunked) time: 69.4943413734436 seconds  

Current file 26: Data/82M Part-2\List # 26_1,036,268 Contacts New Project 82 Million Part-2.xlsx  
86.67% complete  
pandas (chunked) time: 358.22024607658386 seconds  

Current file 27: Data/82M Part-2\List # 27_1,035,004 Contacts New Project 82 Million Part-2.xlsx  
90.00% complete  
pandas (chunked) time: 71.86778497695923 seconds  

Current file 28: Data/82M Part-2\List # 28_1,036,350 Contacts New Project 82 Million Part-2.xlsx  
93.33% complete  
pandas (chunked) time: 75.68091154098511 seconds  

Current file 29: Data/82M Part-2\List # 29_1,036,644 Contacts New Project 82 Million Part-2.xlsx  
96.67% complete  
pandas (chunked) time: 69.55584859848022 seconds  

Current file 30: Data/82M Part-2\List # 30_1,037,204 Contacts New Project 82 Million Part-2.xlsx  
100.00% complete  
pandas (chunked) time: 70.89552235603333 seconds  


### Folder File Search

All files processed in 278.64 seconds.

Current time to beat with traditional:

Files in folder: 30
Search term: plumb
Output folder already exists: Data/82M Part-2/plumb_results

Current file 1: Data/82M Part-2\list_xl 1.csv
File processed in: 0.06 seconds
3.33% complete

Current file 2: Data/82M Part-2\list_xl 10.csv
File processed in: 171.40 seconds
6.67% complete

Current file 3: Data/82M Part-2\list_xl 11.csv

In [ ]:
search_files("Data/82M Part-2", "plumb")

### Logs:

Files in folder: 30
Search term: plumb
Creating output folder: Data/82M Part-2/plumb_results

Current file 1: Data/82M Part-2\list_xl 1.csv
File processed in: 0.01 seconds
3.33% complete

Current file 2: Data/82M Part-2\list_xl 10.csv
File processed in: 11.31 seconds
6.67% complete

Current file 3: Data/82M Part-2\list_xl 11.csv
File processed in: 11.68 seconds
10.00% complete

Current file 4: Data/82M Part-2\list_xl 12.csv
File processed in: 10.25 seconds
13.33% complete

Current file 5: Data/82M Part-2\list_xl 13.csv
File processed in: 10.53 seconds
16.67% complete

Current file 6: Data/82M Part-2\list_xl 14.csv
File processed in: 10.74 seconds
20.00% complete

Current file 7: Data/82M Part-2\list_xl 15.csv
File processed in: 10.59 seconds
23.33% complete

Current file 8: Data/82M Part-2\list_xl 16.csv
File processed in: 10.48 seconds
26.67% complete

Current file 9: Data/82M Part-2\list_xl 17.csv
File processed in: 10.79 seconds
30.00% complete

Current file 10: Data/82M Part-2\list_xl 18.csv
File processed in: 12.38 seconds
33.33% complete

Current file 11: Data/82M Part-2\list_xl 19.csv
File processed in: 10.81 seconds
36.67% complete

Current file 12: Data/82M Part-2\list_xl 2.csv
File processed in: 10.59 seconds
40.00% complete

Current file 13: Data/82M Part-2\list_xl 20.csv
File processed in: 10.33 seconds
43.33% complete

Current file 14: Data/82M Part-2\list_xl 21.csv
File processed in: 10.10 seconds
46.67% complete

Current file 15: Data/82M Part-2\list_xl 22.csv
File processed in: 9.91 seconds
50.00% complete

Current file 16: Data/82M Part-2\list_xl 23.csv
File processed in: 9.86 seconds
53.33% complete

Current file 17: Data/82M Part-2\list_xl 24.csv
File processed in: 10.00 seconds
56.67% complete

Current file 18: Data/82M Part-2\list_xl 25.csv
File processed in: 10.07 seconds
60.00% complete

Current file 19: Data/82M Part-2\list_xl 26.csv
File processed in: 10.09 seconds
63.33% complete

Current file 20: Data/82M Part-2\list_xl 27.csv
File processed in: 10.13 seconds
66.67% complete

Current file 21: Data/82M Part-2\list_xl 28.csv
File processed in: 9.94 seconds
70.00% complete

Current file 22: Data/82M Part-2\list_xl 29.csv
File processed in: 10.11 seconds
73.33% complete

Current file 23: Data/82M Part-2\list_xl 3.csv
File processed in: 9.91 seconds
76.67% complete

Current file 24: Data/82M Part-2\list_xl 30.csv
File processed in: 10.33 seconds
80.00% complete

Current file 25: Data/82M Part-2\list_xl 4.csv
File processed in: 10.13 seconds
83.33% complete

Current file 26: Data/82M Part-2\list_xl 5.csv
C:\Users\3than\AppData\Local\Temp\ipykernel_16532\1216399938.py:28: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
File processed in: 11.04 seconds
86.67% complete

Current file 27: Data/82M Part-2\list_xl 6.csv
File processed in: 10.09 seconds
90.00% complete

Current file 28: Data/82M Part-2\list_xl 7.csv
File processed in: 10.32 seconds
93.33% complete

Current file 29: Data/82M Part-2\list_xl 8.csv
File processed in: 9.98 seconds
96.67% complete

Current file 30: Data/82M Part-2\list_xl 9.csv
File processed in: 10.12 seconds
100.00% complete


All files processed

## 500 Million B2C Leads Database
- 2 excel files, 2 csv.
    - Manually saved each excel file as a csv.
    - Full database.xlsx has two sheets, one with unique values, but it only contains addresses and names.

### Folder File Search

All files processed in 5.83 seconds.

In [18]:
search_files("Data/500 Million B2C Leads Database", "plumb")

Files in folder: 4
Search term: plumb
Creating output folder: Data/500 Million B2C Leads Database/plumb_results

Current file 1: Data/500 Million B2C Leads Database\500K CEO Owner Lead.csv
File processed in: 0.25 seconds
25.00% complete

Current file 2: Data/500 Million B2C Leads Database\Full database.csv


C:\Users\3than\AppData\Local\Temp\ipykernel_21732\1140417957.py:51: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks:


File processed in: 4.66 seconds
50.00% complete

Current file 3: Data/500 Million B2C Leads Database\Tech CEO.csv
File processed in: 0.15 seconds
75.00% complete

Current file 4: Data/500 Million B2C Leads Database\Tech President 5.csv
File processed in: 0.76 seconds
100.00% complete


All files processed in 5.83 seconds.



## Apollo
- Only contains csv files
- 1 and 3 are unsearchable with given program

In [28]:
import os
search_term = "plumb"
file = "Data/Apollo/Apollo 200 Million 1_3.csv"
output_file = f"Data/Apollo/plumb_results/{search_term}_{os.path.basename(file)}"


csv.field_size_limit(10**8)
chunks = load_chunks(file)

In [29]:
search_chunks(chunks, search_term, "Data/Apollo/plumb_results", file)

61
Filtered data saved to Data/Apollo/plumb_results/plumb_Apollo 200 Million 1_3.csv.csv


In [ ]:
import pandas as pd
import csv
search_term = "plumb"
df = pd.read_csv("Data/Apollo/Apollo 200 Million 3_3.csv", chunksize=100000, sep='\t', on_bad_lines='skip', encoding='utf-8', engine='python', quoting=csv.QUOTE_NONE)

for chunk in df:
    # print(chunk.dtypes)
    # print(chunk.columns)
    string_columns = chunk.select_dtypes(include=["object", "string"]).columns
    # print(string_columns)
    mask = chunk[string_columns].apply(lambda col: col.str.contains(search_term, case=False, na=False))
    filtered_chunk = chunk[mask.any(axis=1)]
    # print(filtered_chunk)

## Crypto Emails
- 1 subfolder
    - Contains txt files
- 4 excel files
- 1 txt

It appears this database contains no phone number references.

### Excel to CSV

In [ ]:
xcl_to_csv('Data/Crypto Emails')

Excel to CSV Output:

Files in folder: 4

Current file 1: Data/Crypto Emails\Crypto-email-list-1-million.xlsx
File processed in: 30.01538610458374 seconds
25.00% complete

Current file 2: Data/Crypto Emails\Crypto-email-list-400k.xlsx
File processed in: 10.46195650100708 seconds
50.00% complete

Current file 3: Data/Crypto Emails\Crypto-users-coinmama-1million.xlsx
File processed in: 8.842056035995483 seconds
75.00% complete

Current file 4: Data/Crypto Emails\Crypto-users-coinmama-500k.xlsx
File processed in: 4.439075946807861 seconds
100.00% complete

In [ ]:
import time, os, glob
import pandas as pd

def txt_concat(folder_file_path):
    # "/*.txt" returns all txt files in folder directory
    # "/**/*.txt" and recursive = True returns all txt files in folder and sub-folder directory
    files = glob.glob(folder_file_path + "/**/*.txt", recursive=True)
    print(f"Files in folder: {len(files)}\n")

    # Add index start path
    
    for index, file in enumerate(files):
        
        print(f'Current file {index + 1}: {file}')
        percent = (index + 1) / len(files) * 100

        start_time = time.time()

        # Load txt file in pandas
        df = pd.read_csv(file, delimiter='\t', header=None)
        if index == 0:
            combined_df = df
        else:
            combined_df = pd.concat([combined_df, df], ignore_index=True)
        print(f"Current length: {len(combined_df)}")
        end_time = time.time()
        print(f"File processed in: {end_time - start_time:.2f} seconds")
        print(f'{percent:.2f}% complete\n')
    print(f"Combined length: {len(combined_df)}")
    combined_df.to_csv(f'{folder_file_path}/combined_txt.csv', index=False)


txt_concat('Data/Crypto Emails')

### Folder File Search

All files processed in 11.50 seconds.

In [17]:
search_files("Data/Crypto Emails", "plumb")

Files in folder: 5
Search term: plumb
Creating output folder: Data/Crypto Emails/plumb_results

Current file 1: Data/Crypto Emails\combined_txt.csv
File processed in: 5.19 seconds
20.00% complete

Current file 2: Data/Crypto Emails\Crypto-email-list-1-million.csv
File processed in: 3.70 seconds
40.00% complete

Current file 3: Data/Crypto Emails\Crypto-email-list-400k.csv
File processed in: 1.36 seconds
60.00% complete

Current file 4: Data/Crypto Emails\Crypto-users-coinmama-1million.csv
File processed in: 0.84 seconds
80.00% complete

Current file 5: Data/Crypto Emails\Crypto-users-coinmama-500k.csv
File processed in: 0.42 seconds
100.00% complete


All files processed in 11.50 seconds.



## Ecom Owners 100k
- All but 1 files are are CSV.

In [11]:
xcl_to_csv('Data/Ecom Owners 100k')

Files in folder: 1

Current file 1: Data/Ecom Owners 100k\Ecom Sample.xlsx
File processed in: 0.01 seconds
100.00% complete



All files processed in 1.12 seconds. (Search)

In [16]:
search_files("Data/Ecom Owners 100k", "plumb")

Files in folder: 94
Search term: plumb
Output folder already exists: Data/Ecom Owners 100k/plumb_results

Current file 1: Data/Ecom Owners 100k\Copy of 10.csv
File processed in: 0.02 seconds
1.06% complete

Current file 2: Data/Ecom Owners 100k\Copy of 106.csv
File processed in: 0.01 seconds
2.13% complete

Current file 3: Data/Ecom Owners 100k\Copy of 11.csv
File processed in: 0.01 seconds
3.19% complete

Current file 4: Data/Ecom Owners 100k\Copy of 12.csv
File processed in: 0.01 seconds
4.26% complete

Current file 5: Data/Ecom Owners 100k\Copy of 13.csv
File processed in: 0.01 seconds
5.32% complete

Current file 6: Data/Ecom Owners 100k\Copy of 14.csv
File processed in: 0.01 seconds
6.38% complete

Current file 7: Data/Ecom Owners 100k\Copy of 15.csv
File processed in: 0.01 seconds
7.45% complete

Current file 8: Data/Ecom Owners 100k\Copy of 16.csv
File processed in: 0.01 seconds
8.51% complete

Current file 9: Data/Ecom Owners 100k\Copy of 17.csv
File processed in: 0.01 seconds


## Linkedin L-Series
- All files types are excel files.

### Excel to CSV

In [ ]:
xcl_to_csv('Data/LinkedIn L-Series', 46)

# L42.xlsx throws error No current data in set


Files in folder: 50

Current file 1: Data/LinkedIn L-Series\L1.xlsx
File processed in: 76.99 seconds
2.00% complete

Current file 2: Data/LinkedIn L-Series\L10.xlsx
File processed in: 200.09 seconds
4.00% complete

Current file 3: Data/LinkedIn L-Series\L11.xlsx
File processed in: 196.32 seconds
6.00% complete

Current file 4: Data/LinkedIn L-Series\L12.xlsx
File processed in: 209.44 seconds
8.00% complete

Current file 5: Data/LinkedIn L-Series\L13.xlsx
File processed in: 192.60 seconds
10.00% complete

Current file 6: Data/LinkedIn L-Series\L14.xlsx
File processed in: 203.06 seconds
12.00% complete

Current file 7: Data/LinkedIn L-Series\L15.xlsx
File processed in: 188.07 seconds
14.00% complete

Current file 8: Data/LinkedIn L-Series\L16.xlsx
File processed in: 155.18 seconds
16.00% complete

Current file 9: Data/LinkedIn L-Series\L17.xlsx
File processed in: 157.73 seconds
18.00% complete

Current file 10: Data/LinkedIn L-Series\L18.xlsx
File processed in: 166.32 seconds
20.00% complete

Current file 11: Data/LinkedIn L-Series\L19.xlsx
File processed in: 175.96 seconds
22.00% complete

Current file 12: Data/LinkedIn L-Series\L20.xlsx
File processed in: 181.77 seconds
24.00% complete

Current file 13: Data/LinkedIn L-Series\L21-example.xlsx
File processed in: 0.03 seconds
26.00% complete

Current file 14: Data/LinkedIn L-Series\L21.xlsx
File processed in: 137.50 seconds
28.00% complete

Current file 15: Data/LinkedIn L-Series\L22.xlsx
File processed in: 165.69 seconds
30.00% complete

Current file 16: Data/LinkedIn L-Series\L23.xlsx
File processed in: 175.81 seconds
32.00% complete

Current file 17: Data/LinkedIn L-Series\L24.xlsx
File processed in: 174.03 seconds
34.00% complete

Current file 18: Data/LinkedIn L-Series\L25.xlsx
File processed in: 175.30 seconds
36.00% complete

Current file 19: Data/LinkedIn L-Series\L26.xlsx
File processed in: 151.89 seconds
38.00% complete

Current file 20: Data/LinkedIn L-Series\L27.xlsx
File processed in: 155.44 seconds
40.00% complete

Current file 21: Data/LinkedIn L-Series\L28.xlsx
File processed in: 174.51 seconds
42.00% complete

Current file 22: Data/LinkedIn L-Series\L29.xlsx
File processed in: 179.15 seconds
44.00% complete

Current file 23: Data/LinkedIn L-Series\L30.xlsx
File processed in: 162.07 seconds
46.00% complete

Current file 24: Data/LinkedIn L-Series\L31.xlsx
File processed in: 153.16 seconds
48.00% complete

Current file 25: Data/LinkedIn L-Series\L32.xlsx
File processed in: 158.17 seconds
50.00% complete

Current file 26: Data/LinkedIn L-Series\L33.xlsx
File processed in: 172.82 seconds
52.00% complete

Current file 27: Data/LinkedIn L-Series\L34.xlsx
File processed in: 175.43 seconds
54.00% complete

Current file 28: Data/LinkedIn L-Series\L35.xlsx
File processed in: 173.78 seconds
56.00% complete

Current file 29: Data/LinkedIn L-Series\L36.xlsx
File processed in: 143.09 seconds
58.00% complete

Current file 30: Data/LinkedIn L-Series\L37.xlsx
File processed in: 160.20 seconds
60.00% complete

Current file 31: Data/LinkedIn L-Series\L38.xlsx
File processed in: 170.50 seconds
62.00% complete

Current file 32: Data/LinkedIn L-Series\L39.xlsx
File processed in: 175.98 seconds
64.00% complete

Current file 33: Data/LinkedIn L-Series\L40.xlsx
File processed in: 170.10 seconds
66.00% complete

Current file 34: Data/LinkedIn L-Series\L41.xlsx
File processed in: 147.84 seconds
68.00% complete

File 35 unable to convert  
  
Current file 36: Data/LinkedIn L-Series\L43.xlsx
File processed in: 173.73 seconds
72.00% complete

Current file 37: Data/LinkedIn L-Series\L45.xlsx
File processed in: 168.84 seconds
74.00% complete

Current file 38: Data/LinkedIn L-Series\L46.xlsx
File processed in: 168.56 seconds
76.00% complete

Current file 39: Data/LinkedIn L-Series\L47.xlsx
File processed in: 164.51 seconds
78.00% complete

Current file 40: Data/LinkedIn L-Series\L5.xlsx
File processed in: 254.51 seconds
80.00% complete

Current file 41: Data/LinkedIn L-Series\L51.xlsx
File processed in: 209.33 seconds
82.00% complete

Current file 42: Data/LinkedIn L-Series\L52.xlsx
File processed in: 157.38 seconds
84.00% complete

Current file 43: Data/LinkedIn L-Series\L53.xlsx
File processed in: 171.46 seconds
86.00% complete

Current file 44: Data/LinkedIn L-Series\L54.xlsx
File processed in: 244.64 seconds
88.00% complete

Current file 45: Data/LinkedIn L-Series\L55.xlsx
File processed in: 182.58 seconds
90.00% complete

Current file 46: Data/LinkedIn L-Series\L57.xlsx
File processed in: 149.84 seconds
92.00% complete

Current file 47: Data/LinkedIn L-Series\L6.xlsx
File processed in: 152.45 seconds
94.00% complete

Current file 48: Data/LinkedIn L-Series\L7.xlsx
File processed in: 162.29 seconds
96.00% complete

Current file 49: Data/LinkedIn L-Series\L8.xlsx
File processed in: 159.26 seconds
98.00% complete

Current file 50: Data/LinkedIn L-Series\L9.xlsx
File processed in: 180.14 seconds
100.00% complete

### Search FIle Folder

All files processed in 1208.30 seconds.

In [ ]:
search_files("Data/Linkedin L-Series", "plumb")

## Python CSV Processing Folder
- 1 CSV file
- 1 Excel file, manually converted to CSV.

All files processed in 96.21 seconds.

In [ ]:
search_files("Data/Python CSV Proccesing Folder", "plumb")

## Real Estate Agents
- 1 CSV
- 1 Excel, manually converted to CSV.

All files processed in 8.67 seconds.

In [6]:
search_files("Data/Real Estate Agents", "plumb")

Files in folder: 2
Search term: plumb
Creating output folder: Data/Real Estate Agents/plumb_results

Current file 1: Data/Real Estate Agents\Copy of Real-Estate-175000.csv
File processed in: 1.43 seconds
50.00% complete

Current file 2: Data/Real Estate Agents\Copy of Realtors-Email -860.000.csv
File processed in: 7.24 seconds
100.00% complete


All files processed in 8.67 seconds.



## States Divided
- All files types are CSV.

### File Folder Search

All files processed in 2407.44 seconds.

In [ ]:
search_files("Data/States Divided", "plumb")

## United States

### Blank Emails

All files processed in 502.05 seconds.


In [ ]:
search_files("Data/United States/blank_email_united_states", "plumb")

### Business Emails

All files processed in 235.40 seconds.

In [2]:
search_files("Data/United States/business_email_united_states", "plumb")

Files in folder: 11
Search term: plumb
Creating output folder: Data/United States/business_email_united_states/plumb_results

Current file 1: Data/United States/business_email_united_states\business_email_united_states_0.csv
File processed in: 22.45 seconds
9.09% complete

Current file 2: Data/United States/business_email_united_states\business_email_united_states_1.csv
File processed in: 22.02 seconds
18.18% complete

Current file 3: Data/United States/business_email_united_states\business_email_united_states_2.csv
File processed in: 21.45 seconds
27.27% complete

Current file 4: Data/United States/business_email_united_states\business_email_united_states_3.csv
File processed in: 28.51 seconds
36.36% complete

Current file 5: Data/United States/business_email_united_states\business_email_united_states_4.csv
File processed in: 21.25 seconds
45.45% complete

Current file 6: Data/United States/business_email_united_states\business_email_united_states_5.csv
File processed in: 27.06 second

### Personal Emails

All files processed in 377.99 seconds.

In [3]:
search_files("Data/United States/personal_email_united_states", "plumb")

Files in folder: 18
Search term: plumb
Creating output folder: Data/United States/personal_email_united_states/plumb_results

Current file 1: Data/United States/personal_email_united_states\personal_email_united_states-0001.csv
File processed in: 22.12 seconds
5.56% complete

Current file 2: Data/United States/personal_email_united_states\personal_email_united_states-0002.csv
File processed in: 28.79 seconds
11.11% complete

Current file 3: Data/United States/personal_email_united_states\personal_email_united_states-0003.csv
File processed in: 23.43 seconds
16.67% complete

Current file 4: Data/United States/personal_email_united_states\personal_email_united_states-0004.csv
File processed in: 22.98 seconds
22.22% complete

Current file 5: Data/United States/personal_email_united_states\personal_email_united_states-0005.csv
File processed in: 25.22 seconds
27.78% complete

Current file 6: Data/United States/personal_email_united_states\personal_email_united_states-0006.csv
File processe

## Zoom Info 70 Million
This is the messiest of the bunch
- 2 subfolders
- All files types are excel files.

### File Folder Search

All files processed in 439.01 seconds.

In [4]:
search_files("Data/Zoom Info 70 Million", "plumb")

Files in folder: 44
Search term: plumb
Creating output folder: Data/Zoom Info 70 Million/plumb_results

Current file 1: Data/Zoom Info 70 Million\List # 03_1,038,935 Contacts New Project 143 Million Part-2.csv
File processed in: 10.17 seconds
2.27% complete

Current file 2: Data/Zoom Info 70 Million\List # 05_1,044,894 Contacts New Project 143 Million Part-2.csv
File processed in: 10.35 seconds
4.55% complete

Current file 3: Data/Zoom Info 70 Million\List # 08_1,039,704 Contacts New Project 143 Million Part-2.csv
File processed in: 9.90 seconds
6.82% complete

Current file 4: Data/Zoom Info 70 Million\List # 10_1,038,021 Contacts New Project 143 Million Part-2.csv
File processed in: 10.78 seconds
9.09% complete

Current file 5: Data/Zoom Info 70 Million\List # 11_1,038,212 Contacts New Project 143 Million Part-2.csv
File processed in: 11.19 seconds
11.36% complete

Current file 6: Data/Zoom Info 70 Million\List # 12_1,031,401 Contacts New Project 143 Million Part-2.csv
File processed 

### Excel to CSV

In [ ]:
xcl_to_csv('Data/Zoom Info 70 Million', 35)
# file 3 corrupted: Data/Zoom Info 70 Million\List # 07_1,015,210 Contacts New Project 143 Million Part-2
# file 34 corrupted: Data/Zoom Info 70 Million\List # 54_1,044,327 Contacts New Project 143 Million Part-2.xlsx

Files in folder: 46

Current file 1: Data/Zoom Info 70 Million\List # 03_1,038,935 Contacts New Project 143 Million Part-2.xlsx
File processed in: 109.35 seconds
2.17% complete

Current file 2: Data/Zoom Info 70 Million\List # 05_1,044,894 Contacts New Project 143 Million Part-2.xlsx
File processed in: 112.27 seconds
4.35% complete

Current file 4: Data/Zoom Info 70 Million\List # 08_1,039,704 Contacts New Project 143 Million Part-2.xlsx
File processed in: 107.61 seconds
8.70% complete

Current file 5: Data/Zoom Info 70 Million\List # 10_1,038,021 Contacts New Project 143 Million Part-2.xlsx
File processed in: 143.19 seconds
10.87% complete

Current file 6: Data/Zoom Info 70 Million\List # 11_1,038,212 Contacts New Project 143 Million Part-2.xlsx
File processed in: 92.94 seconds
13.04% complete

Current file 7: Data/Zoom Info 70 Million\List # 12_1,031,401 Contacts New Project 143 Million Part-2.xlsx
File processed in: 87.29 seconds
15.22% complete

Current file 8: Data/Zoom Info 70 Million\List # 13_1,041,488 Contacts New Project 143 Million Part-2.xlsx
File processed in: 87.37 seconds
17.39% complete

Current file 9: Data/Zoom Info 70 Million\List # 14_1,029,664 Contacts New Project 143 Million Part-2.xlsx
File processed in: 83.20 seconds
19.57% complete

Current file 10: Data/Zoom Info 70 Million\List # 15_1.033.871 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.90 seconds
21.74% complete

Current file 11: Data/Zoom Info 70 Million\List # 18_1,044,132 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.51 seconds
23.91% complete

Current file 12: Data/Zoom Info 70 Million\List # 19_1,047,627 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.57 seconds
26.09% complete

Current file 13: Data/Zoom Info 70 Million\List # 20_1,042,825 Contacts New Project 143 Million Part-2.xlsx
File processed in: 82.15 seconds
28.26% complete

Current file 14: Data/Zoom Info 70 Million\List # 21_1,045,060 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.81 seconds
30.43% complete

Current file 15: Data/Zoom Info 70 Million\List # 22_1,045,372 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.60 seconds
32.61% complete

Current file 16: Data/Zoom Info 70 Million\List # 24_1,032,524 Contacts New Project 143 Million Part-2.xlsx
File processed in: 80.20 seconds
34.78% complete

Current file 17: Data/Zoom Info 70 Million\List # 25_1,039,994 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.93 seconds
36.96% complete

Current file 18: Data/Zoom Info 70 Million\List # 26_1,038,162 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.25 seconds
39.13% complete

Current file 19: Data/Zoom Info 70 Million\List # 27_1,044,322 Contacts New Project 143 Million Part-2.xlsx
File processed in: 99.59 seconds
41.30% complete

Current file 20: Data/Zoom Info 70 Million\List # 28_1,042,449 Contacts New Project 143 Million Part-2.xlsx
File processed in: 107.31 seconds
43.48% complete

Current file 21: Data/Zoom Info 70 Million\List # 29_1,044,369 Contacts New Project 143 Million Part-2.xlsx
File processed in: 107.43 seconds
45.65% complete

Current file 22: Data/Zoom Info 70 Million\List # 30_1,044,661 Contacts New Project 143 Million Part-2.xlsx
File processed in: 108.49 seconds
47.83% complete

Current file 23: Data/Zoom Info 70 Million\List # 31_1,044,073 Contacts New Project 143 Million Part-2.xlsx
File processed in: 107.33 seconds
50.00% complete

Current file 24: Data/Zoom Info 70 Million\List # 32_1,042,642 Contacts New Project 143 Million Part-2.xlsx
File processed in: 107.64 seconds
52.17% complete

Current file 25: Data/Zoom Info 70 Million\List # 33_1,046,945 Contacts New Project 143 Million Part-2.xlsx
File processed in: 109.14 seconds
54.35% complete

Current file 26: Data/Zoom Info 70 Million\List # 34_1,045,271 Contacts New Project 143 Million Part-2.xlsx
File processed in: 111.35 seconds
56.52% complete

Current file 27: Data/Zoom Info 70 Million\List # 36_1,044,769 Contacts New Project 143 Million Part-2.xlsx
File processed in: 94.71 seconds
58.70% complete

Current file 28: Data/Zoom Info 70 Million\List # 44_1,046,923 Contacts New Project 143 Million Part-2.xlsx
File processed in: 82.25 seconds
60.87% complete

Current file 29: Data/Zoom Info 70 Million\List # 45_1,039,283 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.83 seconds
63.04% complete

Current file 30: Data/Zoom Info 70 Million\List # 47_1,047,585 Contacts New Project 143 Million Part-2.xlsx
File processed in: 82.67 seconds
65.22% complete

Current file 31: Data/Zoom Info 70 Million\List # 48_1,047,965 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.83 seconds
67.39% complete

Current file 32: Data/Zoom Info 70 Million\List # 49_1,045,427 Contacts New Project 143 Million Part-2.xlsx
File processed in: 82.31 seconds
69.57% complete

Current file 33: Data/Zoom Info 70 Million\List # 51_1,046,432 Contacts New Project 143 Million Part-2.xlsx
File processed in: 83.70 seconds
71.74% complete

Broke at file 34

Current file 35: Data/Zoom Info 70 Million\List # 55_1,045,469 Contacts New Project 143 Million Part-2.xlsx
File processed in: 78.62 seconds
76.09% complete

Current file 36: Data/Zoom Info 70 Million\List # 57_1,047,043 Contacts New Project 143 Million Part-2.xlsx
File processed in: 102.87 seconds
78.26% complete

Current file 37: Data/Zoom Info 70 Million\List # 58_1,045,474 Contacts New Project 143 Million Part-2.xlsx
File processed in: 95.66 seconds
80.43% complete

Current file 38: Data/Zoom Info 70 Million\List # 59_1,045,183 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.86 seconds
82.61% complete

Current file 39: Data/Zoom Info 70 Million\List # 60_1,043,418 Contacts New Project 143 Million Part-2.xlsx
File processed in: 83.86 seconds
84.78% complete

Current file 40: Data/Zoom Info 70 Million\List # 61_1,047,376 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.53 seconds
86.96% complete

Current file 41: Data/Zoom Info 70 Million\List # 62_1,046,659 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.98 seconds
89.13% complete

Current file 42: Data/Zoom Info 70 Million\List # 63_1,046,275 Contacts New Project 143 Million Part-2.xlsx
File processed in: 87.41 seconds
91.30% complete

Current file 43: Data/Zoom Info 70 Million\List # 64_1,047,957 Contacts New Project 143 Million Part-2.xlsx
File processed in: 81.83 seconds
93.48% complete

Current file 44: Data/Zoom Info 70 Million\List # 65_1,046,087 Contacts New Project 143 Million Part-2.xlsx
File processed in: 93.18 seconds
95.65% complete

Current file 45: Data/Zoom Info 70 Million\List # 66_1,046,746 Contacts New Project 143 Million Part-2.xlsx
File processed in: 107.09 seconds
97.83% complete

Current file 46: Data/Zoom Info 70 Million\List # 68_1,046,853 Contacts New Project 143 Million Part-2.xlsx
File processed in: 108.06 seconds
100.00% complete

### LeadRocks
- 23 subfolders
- 80 csv

Goal: Fiqure out if data is duplicated or seperated out into a master list, etc.
Narrow it down to 1 lead rocks source, if that is in any of the zoom info contacts.

Figure out where it's coming from***

Narrow down whole folder to 1 lead file. # include source to identify duplicates.


169 files searched in 3.48 seconds

In [ ]:
# Define the path to the LeadRocks folder
leadrocks_path = "Data/Zoom Info 70 Million/LeadRocks"

search_files(leadrocks_path, 'plumb', subfolder_search=True)

Files in folder: 169
Search term: plumb
Creating output folder: Data/Zoom Info 70 Million/LeadRocks/plumb_results

Current file 1: Data/Zoom Info 70 Million/LeadRocks\leadrocks_business_coach_2023_07_17.csv
File processed in: 0.02 seconds
0.59% complete

Current file 2: Data/Zoom Info 70 Million/LeadRocks\leadrocks_business_coach_ceo_2023_07_17.csv
File processed in: 0.02 seconds
1.18% complete

Current file 3: Data/Zoom Info 70 Million/LeadRocks\leadrocks_business_coach_founder_2023_07_17.csv
File processed in: 0.02 seconds
1.78% complete

Current file 4: Data/Zoom Info 70 Million/LeadRocks\leadrocks_business_coach_linkedin_1_2023_07_17.csv
File processed in: 0.01 seconds
2.37% complete

Current file 5: Data/Zoom Info 70 Million/LeadRocks\leadrocks_california_agency_owners_2023_07_17.csv
File processed in: 0.02 seconds
2.96% complete

Current file 6: Data/Zoom Info 70 Million/LeadRocks\leadrocks_cannabis_1_2023_07_17.csv
File processed in: 0.05 seconds
3.55% complete

Current file 7: 

### Ninja Leads
- 101 subfolders

Backburner data source, prioritize raw data, clean as you go. End product should be able to search keyword: Output nice db.

185 files processed in 5 seconds

In [12]:
leadrocks_path = "Data/Zoom Info 70 Million/Ninja Leads"
output_folder = "Data/Zoom Info 70 Million/Ninja_leads_plumb_results"
search_files(leadrocks_path, 'plumb', subfolder_search=True, output_folder=output_folder, file_index_start=50)

Files in folder: 185
Search term: plumb
Output folder already exists: Data/Zoom Info 70 Million/Ninja_leads_plumb_results

Current file 50: Data/Zoom Info 70 Million/Ninja Leads\Fitness Center\Fitness Center 1.csv
File processed in: 0.04 seconds
27.03% complete

Current file 51: Data/Zoom Info 70 Million/Ninja Leads\Fitness Center\Fitness Center 2.csv
File processed in: 0.02 seconds
27.57% complete

Current file 52: Data/Zoom Info 70 Million/Ninja Leads\Fitness Center\Fitness Center 3.csv
File processed in: 0.02 seconds
28.11% complete

Current file 53: Data/Zoom Info 70 Million/Ninja Leads\Fitness Center\Fitness Center 4.csv
File processed in: 0.02 seconds
28.65% complete

Current file 54: Data/Zoom Info 70 Million/Ninja Leads\Floor Installation\Floor Installation - 7_19_23.csv
File processed in: 0.02 seconds
29.19% complete

Current file 55: Data/Zoom Info 70 Million/Ninja Leads\Floor Installation\Floor Installation - 7_21_23.csv
File processed in: 0.02 seconds
29.73% complete

Curre

# File Aggregation and Cleaning

## General Code:
Go through each file in output folder and return column names, types, and disparities.

In [127]:
import glob
import pandas as pd
from collections import defaultdict

def column_search(folder_path):
    """
    input: folder path
    output: dictionary with missing and extra columns for each file
    """
    files = glob.glob(folder_path + "/*.csv")
    print(f"Found {len(files)} files in folder.\n")
    column_names = {}
    all_columns = set()

    for file in files:
        try:
            df = pd.read_csv(file, on_bad_lines='skip', encoding='utf-8')
            # Ensure duplicate columns are handled
            df_columns = set(df.columns)
            if len(df_columns) != len(df.columns):
                print(f"Warning: Duplicate column names found in {file}.")
            column_names[file] = df_columns
            all_columns.update(df_columns)  # Combine columns directly
        except Exception as e:
            print(f"Error reading {file}: {e}")
            continue

    differences = defaultdict(dict)
    for file_name, columns in column_names.items():
        missing_columns = all_columns - columns
        extra_columns = columns - all_columns
        if missing_columns:
            differences[file_name]['missing'] = missing_columns
        if extra_columns:
            differences[file_name]['extra'] = extra_columns
    
    return differences

def column_list(folder_path, spec_column=None):
    files = glob.glob(folder_path + "/*.csv")
    print(f"Found {len(files)} files in folder.\n")

    for file in files:
                
        try:
            # Read only the first chunk to get the column names
            df = pd.read_csv(file, on_bad_lines='skip', encoding='utf-8')
            # df = next(chunk)
            print(f"File: {file}")
            if spec_column:
                if spec_column in df.columns:
                    print(df[spec_column].head())
            else:
                print(f"Columns: {df.columns.tolist()}\n")
                
        except Exception as e:
            print(f"Error reading {file}: {e}")
            continue


def group_by_missing_columns(differences):
    """
    Group files by their missing columns and print results.
    """
    grouped = defaultdict(list)
    for file, info in differences.items():
        missing_columns = tuple(sorted(info.get("missing", [])))  # Sort columns for consistent grouping
        grouped[missing_columns].append(file)

    print("\nGrouped Missing Columns Report:")
    for missing_columns, files in grouped.items():
        print(f"Missing Columns: {', '.join(missing_columns)}")
        print(f"  Affected Files ({len(files)}):")
        for file in files:
            print(f"    {file}")
        print("\n")


def summarize_differences(differences):
    """
    Print a summary of overall discrepancies across all files.
    """
    file_count = len(differences)
    unique_missing_columns = set()
    total_missing_instances = 0

    for info in differences.values():
        missing = info.get("missing", [])
        unique_missing_columns.update(missing)
        total_missing_instances += len(missing)

    print("\nSummary of Column Disparity Analysis:")
    print(f"  Total Files Analyzed: {file_count}")
    print(f"  Total Unique Missing Columns: {len(unique_missing_columns)}")
    print(f"  Total Missing Column Instances: {total_missing_instances}")
    print("\nUnique Missing Columns:")
    for column in sorted(unique_missing_columns):
        print(f"  - {column}")

def rename_columns(folder_path, column_map):# column_map: {'old column' : 'new_column'}
    """
    input: folder path, dictionary of old column names to new column names
    output: None, modifies files in place
    """
    files = glob.glob(folder_path + "/*.csv")
    print(f"Found {len(files)} files in folder.\n")

    for file in files:
        try:
            df = pd.read_csv(file, on_bad_lines='skip', encoding='utf-8')
            df.rename(columns=column_map, inplace=True)
            df.to_csv(file, index=False)
            print(f"Renamed columns in {file}")
        except Exception as e:
            print(f"Error reading {file}: {e}")
            continue

def remove_columns_with_exact_string(folder_path, target_string="#!$@-"):
    """
    Remove columns from a DataFrame if any cell in that column contains the exact string `#!$@-`.

    Args:
        df (pd.DataFrame): The input DataFrame.
        target_string (str): The string to look for. Default is `#!$@-`.

    Returns:
        pd.DataFrame: A new DataFrame with columns removed if they contain the target string.
    """
    # Identify columns to keep (i.e., those that don't contain the exact target string)
    files = glob.glob(folder_path + "/*.csv")

    for file in files:
        df = pd.read_csv(file, encoding='utf-8')
        
        col1_length = len(df.columns)
        df = df.loc[:, ~df.isin([target_string]).any()]
        col2_length = len(df.columns)

        print(f"Columns removed: {col1_length - col2_length}")
        df.to_csv(file, index=False)

def combine_files(folder_path, file_output_name, output_folder_path="Filtered Output"):
    files = glob.glob(folder_path + "/*.csv")

    combined_df = pd.concat((pd.read_csv(file) for file in files), ignore_index=True)

    print(combined_df.duplicated().sum())
    print(combined_df[combined_df.duplicated()])
    print(len(combined_df))
    # combined_df.to_csv(f"{output_folder_path}/{file_output_name}", index=False)        


### 82M Part-2

82M Part-2: `No Column Difference`

In [31]:
folder_path = "Data/82M Part-2/plumb_results"  # Replace with your folder path containing .csv files
differences = column_search(folder_path)

group_by_missing_columns(differences)

summarize_differences(differences)


Found 3 files in folder.



C:\Users\3than\AppData\Local\Temp\ipykernel_18496\871987284.py:18: DtypeWarning: Columns (26,34,43,44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, on_bad_lines='skip', encoding='utf-8')



Grouped Missing Columns Report:
Missing Columns: _id, _index, _score, _type, job_functions, organization_alexa_ranking, organization_all_possible_domains, organization_angellist_market_tag_ids, organization_angellist_markets, organization_angellist_url, organization_current_technologies, organization_domain, organization_domain_analyzed, organization_domain_status_cd, organization_facebook_url, organization_founded_year, organization_hq_location_city, organization_hq_location_city_with_state_or_country, organization_hq_location_country, organization_hq_location_geojson, organization_hq_location_postal_code, organization_hq_location_state, organization_hq_location_state_with_country, organization_id, organization_industries, organization_keywords, organization_languages, organization_latest_funding_round_amount_long, organization_latest_funding_round_date, organization_latest_funding_stage_cd, organization_linkedin_company_size_tag_ids, organization_linkedin_industry_tag_ids, organizat

C:\Users\3than\AppData\Local\Temp\ipykernel_18496\871987284.py:18: DtypeWarning: Columns (26,34,43,44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, on_bad_lines='skip', encoding='utf-8')


In [ ]:
rename_columns("Data/82M Part-2/plumb_results", {'Reveneu': 'Revenue'})

In [128]:
## Combine all files into master list
combine_files("Data/82M Part-2/plumb_results", "82M_combined_plumb_results.csv")

2
      First Name Middle Name Last Name                Title  \
25410     Daniel         NaN    Salvas     Licensed Plumber   
25597      Larry         NaN    Rosser  Estimating Plumbing   

                Company Name     Mailing Address Primary City Primary State  \
25410  Re/Max Holdings, Inc.  5075 S Syracuse St       Denver            CO   
25597  Re/Max Holdings, Inc.  5075 S Syracuse St       Denver            CO   

      ZIP Code Country         Phone    Web Address              Email  \
25410    80237     USA  303-770-5531  www.remax.com  dsalvas@remax.com   
25597    80237     USA  303-770-5531  www.remax.com  lrosser@remax.com   

      Revenue   Employee                    Industry  \
25410   > $1B  10K - 50K  Real Estate & Construction   
25597   > $1B  10K - 50K  Real Estate & Construction   

                            Sub Industry  
25410  Real Estate Agents and Appraisers  
25597  Real Estate Agents and Appraisers  
26462


### 500 Million B2C Leads Database

500 Million B2C Leads Database `Large Column Difference`

In [4]:
differences = column_search("Data/500 Million B2C Leads Database/plumb_results")


Found 2 files in folder.



In [10]:
column_list("Data/500 Million B2C Leads Database/plumb_results")

# group_by_missing_columns(differences)
# summarize_differences(differences)

Found 2 files in folder.

File: Data/500 Million B2C Leads Database/plumb_results\plumb_500K CEO Owner Lead.csv
Columns: ['First Name', 'Last Name', 'Title', 'Email', 'Email Status', 'Person Linkedin Url', 'Company', 'Website', 'Company Linkedin Url', 'Company Address', 'Industry', 'City', 'State', 'Country']

File: Data/500 Million B2C Leads Database/plumb_results\plumb_Full database.csv
Columns: ['CityURL', 'Facility URL', 'Name', 'Phone', 'Address', 'City', 'State', 'Zip', 'Review', 'Room Type', 'Amenities']



### Apollo

Apollo 1_3 & 3_3 `Same Columns`  
Apollo 2_3 `Tab delimited, different columns`

In [12]:
column_list("Data/Apollo/plumb_results")
differences = column_search("Data/Apollo/plumb_results")

Found 3 files in folder.

File: Data/Apollo/plumb_results\Apollo 200 Million 2_3.csv
Columns: ['person_name\tperson_first_name_unanalyzed\tperson_last_name_unanalyzed\tperson_name_unanalyzed_downcase\tperson_title\tperson_functions\tperson_seniority\tperson_email_status_cd\tperson_extrapolated_email_confidence\tperson_email\tperson_phone\tperson_sanitized_phone\tperson_email_analyzed\tperson_linkedin_url\tperson_detailed_function\tperson_title_normalized\tprimary_title_normalized_for_faceting\tsanitized_organization_name_unanalyzed\tperson_location_city\tperson_location_city_with_state_or_country\tperson_location_state\tperson_location_state_with_country\tperson_location_country\tperson_location_postal_code\tjob_start_date\tcurrent_organization_ids\tmodality\tprospected_by_team_ids\tperson_excluded_by_team_ids\trelavence_boost\tperson_num_linkedin_connections\tperson_location_geojson\tpredictive_scores\tperson_vacuumed_at\trandom\t_index\t_type\t_id\t_score']

File: Data/Apollo/plumb_r

C:\Users\3than\AppData\Local\Temp\ipykernel_7220\2619615475.py:17: DtypeWarning: Columns (26,34,43,44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, on_bad_lines='skip', encoding='utf-8')
C:\Users\3than\AppData\Local\Temp\ipykernel_7220\2619615475.py:17: DtypeWarning: Columns (26,34,43,44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, on_bad_lines='skip', encoding='utf-8')


In [13]:
group_by_missing_columns(differences)
summarize_differences(differences)


Grouped Missing Columns Report:
Missing Columns: _id, _index, _score, _type, job_functions, organization_alexa_ranking, organization_all_possible_domains, organization_angellist_market_tag_ids, organization_angellist_markets, organization_angellist_url, organization_current_technologies, organization_domain, organization_domain_analyzed, organization_domain_status_cd, organization_facebook_url, organization_founded_year, organization_hq_location_city, organization_hq_location_city_with_state_or_country, organization_hq_location_country, organization_hq_location_geojson, organization_hq_location_postal_code, organization_hq_location_state, organization_hq_location_state_with_country, organization_id, organization_industries, organization_keywords, organization_languages, organization_latest_funding_round_amount_long, organization_latest_funding_round_date, organization_latest_funding_stage_cd, organization_linkedin_company_size_tag_ids, organization_linkedin_industry_tag_ids, organizat

### Crypto Emails

email-list-1-million & email-list-400k `same`, small disparity  
  
The rest are only `emails`  


In [14]:
column_list("Data/Crypto Emails/plumb_results")
differences = column_search("Data/Crypto Emails/plumb_results")

Found 5 files in folder.

File: Data/Crypto Emails/plumb_results\plumb_combined_txt.csv
Columns: ['00@mail.iswest.com']

File: Data/Crypto Emails/plumb_results\plumb_Crypto-email-list-1-million.csv
Columns: ['Email', 'Fname', 'Lname', 'address', 'city ', 'zip', 'phone']

File: Data/Crypto Emails/plumb_results\plumb_Crypto-email-list-400k.csv
Columns: ['Email', 'Fname', 'Lname', 'Address', 'City', 'Unnamed: 5']

File: Data/Crypto Emails/plumb_results\plumb_Crypto-users-coinmama-1million.csv
Columns: ['kamilla_hanssen@hotmail.com']

File: Data/Crypto Emails/plumb_results\plumb_Crypto-users-coinmama-500k.csv
Columns: ['kamilla_hanssen@hotmail.com']

Found 5 files in folder.



In [15]:
group_by_missing_columns(differences)
summarize_differences(differences)


Grouped Missing Columns Report:
Missing Columns: Address, City, Email, Fname, Lname, Unnamed: 5, address, city , kamilla_hanssen@hotmail.com, phone, zip
  Affected Files (1):
    Data/Crypto Emails/plumb_results\plumb_combined_txt.csv


Missing Columns: 00@mail.iswest.com, Address, City, Unnamed: 5, kamilla_hanssen@hotmail.com
  Affected Files (1):
    Data/Crypto Emails/plumb_results\plumb_Crypto-email-list-1-million.csv


Missing Columns: 00@mail.iswest.com, address, city , kamilla_hanssen@hotmail.com, phone, zip
  Affected Files (1):
    Data/Crypto Emails/plumb_results\plumb_Crypto-email-list-400k.csv


Missing Columns: 00@mail.iswest.com, Address, City, Email, Fname, Lname, Unnamed: 5, address, city , phone, zip
  Affected Files (2):
    Data/Crypto Emails/plumb_results\plumb_Crypto-users-coinmama-1million.csv
    Data/Crypto Emails/plumb_results\plumb_Crypto-users-coinmama-500k.csv



Summary of Column Disparity Analysis:
  Total Files Analyzed: 5
  Total Unique Missing Columns:

### Ecom Owners 100k

This dataset hardly has any returned data

In [16]:
folder_path = "Data/Ecom Owners 100k/plumb_results"
column_list(folder_path)
differences = column_search(folder_path)

Found 13 files in folder.

File: Data/Ecom Owners 100k/plumb_results\plumb_Copy of 106.csv
Columns: ['Unnamed: 0', 'First Name', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Email', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']

File: Data/Ecom Owners 100k/plumb_results\plumb_Copy of 28.csv
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Email', 'Unnamed: 4', 'Unnamed: 5', 'First name', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17']

File: Data/Ecom Owners 100k/plumb_results\plumb_Copy of 30.csv
Columns: ['Email', 'First name', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

File: Data/Ecom Owners 100k/plumb_results\plumb_Copy of 39.csv
Columns: ['Email', 'First na

In [20]:
group_by_missing_columns(differences)
# summarize_differences(differences)


Grouped Missing Columns Report:
Missing Columns: First name, Unnamed: 1, Unnamed: 10, Unnamed: 17, Unnamed: 18, Unnamed: 19, Unnamed: 20, Unnamed: 21, Unnamed: 22, Unnamed: 23, Unnamed: 24, Unnamed: 25, Unnamed: 26, Unnamed: 27, Unnamed: 28, Unnamed: 29, Unnamed: 30, Unnamed: 31, Unnamed: 32, Unnamed: 33, Unnamed: 34, Unnamed: 35, Unnamed: 36, Unnamed: 37, Unnamed: 38, Unnamed: 39
  Affected Files (1):
    Data/Ecom Owners 100k/plumb_results\plumb_Copy of 106.csv


Missing Columns: First Name, Unnamed: 18, Unnamed: 19, Unnamed: 20, Unnamed: 21, Unnamed: 22, Unnamed: 23, Unnamed: 24, Unnamed: 25, Unnamed: 26, Unnamed: 27, Unnamed: 28, Unnamed: 29, Unnamed: 3, Unnamed: 30, Unnamed: 31, Unnamed: 32, Unnamed: 33, Unnamed: 34, Unnamed: 35, Unnamed: 36, Unnamed: 37, Unnamed: 38, Unnamed: 39, Unnamed: 6
  Affected Files (1):
    Data/Ecom Owners 100k/plumb_results\plumb_Copy of 28.csv


Missing Columns: First Name, Unnamed: 0, Unnamed: 1, Unnamed: 12, Unnamed: 13, Unnamed: 14, Unnamed: 15, U

### LinkedIn L-series

In [116]:
folder_path = "Data/LinkedIn L-Series/plumb_results"
column_list(folder_path, spec_column='Column3')
# differences = column_search(folder_path)

Found 48 files in folder.

File: Data/LinkedIn L-Series/plumb_results\plumb_L1.csv
File: Data/LinkedIn L-Series/plumb_results\plumb_L10.csv
0     Elaine Haughton
1     Joseph Routhier
2         Laron Brown
3      Ankita Agarwal
4    Benjamin Carrick
Name: Column3, dtype: object
File: Data/LinkedIn L-Series/plumb_results\plumb_L11.csv
0      Matthew Flati
1          Bibhuti D
2      Michel Groulx
3      Jeremy Herbst
4    William Tinnell
Name: Column3, dtype: object
File: Data/LinkedIn L-Series/plumb_results\plumb_L12.csv
0    Johnny Laster
1     Robert Lebel
2        Dj Skream
3     Bobby Chaney
4     Casey Rohaus
Name: Column3, dtype: object
File: Data/LinkedIn L-Series/plumb_results\plumb_L13.csv
0          Vikas S
1      Avery Evans
2    Gaurav Mishra
3        Gage Bill
4      Colin Gower
Name: Column3, dtype: object
File: Data/LinkedIn L-Series/plumb_results\plumb_L14.csv
0    Scott Crawford
1     Melvin Parker
2    Ernesto Pinder
3     Danish Juikar
4      John Naranjo
Name: Colum

In [89]:
# group_by_missing_columns(differences)
summarize_differences(differences)


Summary of Column Disparity Analysis:
  Total Files Analyzed: 48
  Total Unique Missing Columns: 70
  Total Missing Column Instances: 2839

Unique Missing Columns:
  -    
  - Column11
  - Column13
  - Column15
  - Column17
  - Column19
  - Column21
  - Column23
  - Column25
  - Column27
  - Column29
  - Column3
  - Column31
  - Column33
  - Column35
  - Column37
  - Column39
  - Column41
  - Column43
  - Column45
  - Column47
  - Column49
  - Column5
  - Column51
  - Column53
  - Column55
  - Column57
  - Column59
  - Column61
  - Column63
  - Column65
  - Column66
  - Column67
  - Column68
  - Column69
  - Column7
  - Column9
  - organization_angellist_url
  - organization_current_technologies
  - organization_domain
  - organization_facebook_url
  - organization_founded_year
  - organization_hq_location_city
  - organization_hq_location_country
  - organization_hq_location_postal_code
  - organization_hq_location_state
  - organization_industries
  - organization_languages
  - orga

In [ ]:
remove_columns_with_exact_string(folder_path, "#!$@-")

In [21]:
import pandas as pd

df = pd.read_csv("Filtered Output/82M_combined_plumb_results.csv")
print(df.columns)
print(len(df))

Index(['First Name', 'Middle Name', 'Last Name', 'Title', 'Company Name',
       'Mailing Address', 'Primary City', 'Primary State', 'ZIP Code',
       'Country', 'Phone', 'Web Address', 'Email', 'Revenue', 'Employee',
       'Industry', 'Sub Industry'],
      dtype='object')
26462


In [3]:
df_filtered = df[df["Mailing Address"].str.contains("plumb", case=False, na=False)]

In [19]:
df_good_addy = []
for column in df_filtered.columns:
    df_temp = df_filtered[df_filtered[column].str.contains('plumb', case=False, na=False)]
    print(f"Len of {column}: {len(df_temp)}")
    if len(df_temp) < 560 and len(df_temp) > 0:
        # print(df_temp[column])
        df_good_addy.append(df_temp)

temp = pd.concat(df_good_addy)
print(len(temp))

df_no_plumb_addy = df[~df["Mailing Address"].str.contains("plumb", case=False, na=False)]
new_df = pd.concat([df_no_plumb_addy, temp])
print(len(new_df))
print(len(df) - len(df_filtered) + len(temp))

new_df.to_csv("Filtered Output/82M_combined_plumb_results_filtered.csv", index=False)

Len of First Name: 0
Len of Middle Name: 0
Len of Last Name: 0
Len of Title: 1
Len of Company Name: 4
Len of Mailing Address: 562
Len of Primary City: 0
Len of Primary State: 0
Len of ZIP Code: 0
Len of Country: 0
Len of Phone: 0
Len of Web Address: 0
Len of Email: 0
Len of Revenue: 0
Len of Employee: 0
Len of Industry: 0
Len of Sub Industry: 0
5
25905
25905


In [3]:
import pandas as pd
df = pd.read_csv("Filtered Output/82M_combined_plumb_results_filtered.csv")
print(len(df))
for column in df.columns:
    df_temp = df[df[column].str.contains('plumb', case=False, na=False)]
    print(f"Len of {column}: {len(df_temp)}")
df_last_name = df[df["Last Name"].str.contains('plumb', case=False, na=False)]

25905
Len of First Name: 0
Len of Middle Name: 0
Len of Last Name: 794
Len of Title: 1476
Len of Company Name: 22384
Len of Mailing Address: 5
Len of Primary City: 0
Len of Primary State: 0
Len of ZIP Code: 0
Len of Country: 0
Len of Phone: 0
Len of Web Address: 16080
Len of Email: 16815
Len of Revenue: 0
Len of Employee: 0
Len of Industry: 0
Len of Sub Industry: 0


In [12]:
keep_cols = []
for column in df_last_name.columns:
    df_temp = df_last_name[df_last_name[column].str.contains('plumb', case=False, na=False)]
    print(f"Len of {column}: {len(df_temp)}")
    if len(df_temp) < 80 and len(df_temp) > 0:
        print(df_temp[column])
        keep_cols.append(df_temp)


last_name_no_plumb = df[~df["Last Name"].str.contains("plumb", case=False, na=False)]
keep_cols.append(last_name_no_plumb)
new_df = pd.concat(keep_cols)
for column in new_df.columns:
    df_temp = new_df[new_df[column].str.contains('plumb', case=False, na=False)]
    print(f"Len of {column}: {len(df_temp)}")

# new_df.to_csv("Filtered Output/82M_plumb.csv", index=False)
print(len(new_df))

Len of First Name: 0
Len of Middle Name: 0
Len of Last Name: 794
Len of Title: 0
Len of Company Name: 1
4032    Denron Plumbing & Hvac Llc
Name: Company Name, dtype: object
Len of Mailing Address: 0
Len of Primary City: 0
Len of Primary State: 0
Len of ZIP Code: 0
Len of Country: 0
Len of Phone: 0
Len of Web Address: 0
Len of Email: 700
Len of Revenue: 0
Len of Employee: 0
Len of Industry: 0
Len of Sub Industry: 0
Len of First Name: 0
Len of Middle Name: 0
Len of Last Name: 1
Len of Title: 1476
Len of Company Name: 22384
Len of Mailing Address: 5
Len of Primary City: 0
Len of Primary State: 0
Len of ZIP Code: 0
Len of Country: 0
Len of Phone: 0
Len of Web Address: 16080
Len of Email: 16116
Len of Revenue: 0
Len of Employee: 0
Len of Industry: 0
Len of Sub Industry: 0
25112
